# Phase 1: Setup & Data Loading
Run this notebook to download the Ireland hourly weather dataset using the Kaggle API.

In [ ]:
!pip install kagglehub pandas pyarrow -q

import kagglehub
import pandas as pd
import glob
import os

project_dir = '..'
os.makedirs(f'{project_dir}/data', exist_ok=True)
os.makedirs(f'{project_dir}/outputs', exist_ok=True)

# Download the Ireland hourly weather dataset
path = kagglehub.dataset_download("dariasvasileva/hourly-weather-data-in-ireland-from-24-stations")
print("Dataset downloaded to:", path)

# Merge all CSVs if multiple exist, or just copy the main one
files = glob.glob(f'{path}/**/*.csv', recursive=True)
dfs = []
for f in files:
    try:
        d = pd.read_csv(f)
        if 'station' not in d.columns:
            d['station'] = os.path.basename(f).replace('.csv', '')
        dfs.append(d)
    except Exception as e:
        print(f"Could not read {f}: {e}")

if dfs:
    df_all = pd.concat(dfs, ignore_index=True)
    df_all.to_parquet(f'{project_dir}/data/ireland_all_stations.parquet', index=False)
    print(f"Merged data saved. Total rows: {len(df_all):,}")

# Phase 2: Data Cleaning & Exploratory Data Analysis

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import yaml
from utils.plot_utils import plot_feature_distributions, plot_correlation_matrix
from src.data_loader import load_data

# Load configuration
with open('../config/config.yaml', 'r') as file:
    config = yaml.safe_load(file)

feature_cols = config['data']['feature_cols']

# Load merged data
df = pd.read_parquet('../data/ireland_all_stations.parquet')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Handle Missing Values
df[feature_cols] = df[feature_cols].fillna(method='ffill', limit=3)
df[feature_cols] = df[feature_cols].fillna(method='bfill', limit=3)

# Drop rows still missing the target variable
df = df.dropna(subset=[config['data']['target_col']])

# Cap extreme outliers for rain
df['rain'] = df['rain'].clip(lower=0, upper=100)

# Save cleaned dataset
df[['date'] + feature_cols].to_parquet('../data/ireland_clean.parquet', index=False)
print(f"Clean dataset saved: {len(df):,} rows")


In [ ]:
# Plotting EDA
plot_feature_distributions(df, feature_cols, '../outputs/feature_distributions.png')
plot_correlation_matrix(df, feature_cols, '../outputs/correlation_matrix.png')